In [1]:
import wmfdata as wmf
import pandas as pd
import numpy as np
from wmfdata import spark,hive
import plotly.express as px

In [3]:
# Query

regional_uniques = wmf.spark.run("""
SELECT 
    date_format(CONCAT(ud.year, '-', LPAD(ud.month, 2, '0'), '-01'), 'yyyy-MM') AS month,
    ud.country,
    SUM(ud.uniques_estimate) AS total_unique_devices,
    cmd.wmf_region
FROM 
    wmf.unique_devices_per_project_family_monthly ud
JOIN 
    gdi.country_meta_data cmd
ON 
    ud.country_code = cmd.country_code_iso_2
WHERE 
    ud.project_family = 'wikipedia'
    AND ud.year in (2023,2022)
    AND ud.month = 11
GROUP BY 
    date_format(CONCAT(ud.year, '-', LPAD(ud.month, 2, '0'), '-01'), 'yyyy-MM'), 
    ud.country, ud.country_code, cmd.wmf_region




""")

# Convert query results into dataframe
df = regional_uniques.copy()
df['month'] = pd.to_datetime(df['month'], format='%Y-%m')

# Convert the unique months to a Series and sort
unique_months = pd.Series(df['month'].dt.to_period('M').unique()).sort_values(ascending=False)

# Initialize an empty DataFrame to hold all results
final_df = pd.DataFrame()

for current_month in unique_months:
    previous_year_month = current_month.to_timestamp() - pd.DateOffset(years=1)

    # Filter the data for the current month and the same month in the previous year
    filtered_df = df[df['month'].isin([current_month.to_timestamp(), previous_year_month])]

    # Pivot table for the filtered data
    df_pivot = filtered_df.pivot_table(index=['country', 'wmf_region'], 
                                   columns='month', 
                                   values='total_unique_devices',
                                   aggfunc='sum').reset_index()
    
    # Storing Initial Values
    df_pivot['current_metric'] = df_pivot[current_month.to_timestamp()]
    df_pivot['previous_metric'] = df_pivot.get(previous_year_month, 0)


    # Calculating Absolute Change
    df_pivot['absolute_change'] = df_pivot['current_metric'] - df_pivot['previous_metric']

    # Adding Current and Previous Metrics
    df_pivot['current_metric'] = df_pivot[current_month.to_timestamp()]
    df_pivot['previous_metric'] = df_pivot.get(previous_year_month, 0)

    # Calculate Regional and Global Totals
    regional_totals = df_pivot.groupby('wmf_region')[['current_metric', 'previous_metric']].sum().reset_index()
    global_current_metric_total = df_pivot['current_metric'].sum()
    global_previous_metric_total = df_pivot['previous_metric'].sum()

    # Merge with Regional Totals and add Global Totals
    df_merged = pd.merge(df_pivot, regional_totals, on='wmf_region', suffixes=('', '_regional_total'))
    df_merged['global_current_metric_total'] = global_current_metric_total
    df_merged['global_previous_metric_total'] = global_previous_metric_total

    # Calculating Proportions and Formula Results
    df_merged['proportion_current_regional'] = df_merged['current_metric'] / df_merged['current_metric_regional_total']
    df_merged['proportion_previous_regional'] = df_merged['previous_metric'] / df_merged['previous_metric_regional_total']
    df_merged['proportion_current_global'] = df_merged['current_metric'] / df_merged['global_current_metric_total']
    df_merged['proportion_previous_global'] = df_merged['previous_metric'] / df_merged['global_previous_metric_total']

    df_merged['formula_result_regional'] = abs(((df_merged['current_metric'] - df_merged['previous_metric']) * df_merged['proportion_current_regional']) +
                                               ((df_merged['proportion_current_regional'] - df_merged['proportion_previous_regional']) * (df_merged['previous_metric'] - df_merged['previous_metric_regional_total'])))
    df_merged['formula_result_global'] = abs(((df_merged['current_metric'] - df_merged['previous_metric']) * df_merged['proportion_current_global']) +
                                             ((df_merged['proportion_current_global'] - df_merged['proportion_previous_global']) * (df_merged['previous_metric'] - df_merged['global_previous_metric_total'])))

    # Calculate the Total Sum of 'formula_result' per Region and Globally
    total_formula_result_regional = df_merged.groupby('wmf_region')['formula_result_regional'].sum().reset_index().rename(columns={'formula_result_regional': 'formula_result_regional_sum'})
    total_formula_result_global = df_merged['formula_result_global'].sum()

    # Merge with total formula results and calculate percentages
    df_merged = pd.merge(df_merged, total_formula_result_regional, on='wmf_region')
    df_merged['formula_result_percentage_regional'] = (df_merged['formula_result_regional'] / df_merged['formula_result_regional_sum']) * 100
    df_merged['formula_result_percentage_global'] = (df_merged['formula_result_global'] / total_formula_result_global) * 100

    # Simple Calculation Columns
    total_abs_change_regional = df_merged.groupby('wmf_region')['absolute_change'].apply(lambda x: x.abs().sum()).reset_index().rename(columns={'absolute_change': 'total_abs_change_regional'})
    total_abs_change_global = df_merged['absolute_change'].abs().sum()

    # Merge the total absolute changes with the main DataFrame
    df_merged = pd.merge(df_merged, total_abs_change_regional, on='wmf_region')

    # Calculate the simple calculation columns as percentages
    df_merged['simple_calculation_regional'] = (df_merged['absolute_change'].abs() / df_merged['total_abs_change_regional']) * 100
    df_merged['simple_calculation_global'] = (df_merged['absolute_change'].abs() / total_abs_change_global) * 100

    # Add a 'month' column to df_merged
    df_merged['month'] = current_month.strftime('%Y-%m')
    
    # Add ranking logic
    df_merged['rank_simple_calculation'] = df_merged.groupby(['month', 'wmf_region'])['simple_calculation_regional'].rank(method='dense', ascending=False)
    df_merged['rank_formula_result'] = df_merged.groupby(['month', 'wmf_region'])['formula_result_percentage_regional'].rank(method='dense', ascending=False)


    # Append the results for this month to the final DataFrame
    final_df = pd.concat([final_df, df_merged])


# Convert 'absolute_change' to its absolute value
final_df['absolute_change'] = final_df['absolute_change'].abs()

# Select required columns
final_df = final_df[['month', 'country', 'wmf_region', 'current_metric', 'previous_metric', 'absolute_change', 
                     'formula_result_percentage_regional', 'simple_calculation_regional', 
                     'rank_simple_calculation', 'rank_formula_result',
                     'simple_calculation_global', 'formula_result_percentage_global']]


# Sort the DataFrame by 'month', 'wmf_region', and 'absolute_change'
final_df = final_df.sort_values(by=['month', 'wmf_region', 'absolute_change'], ascending=[False, True, False])

# Save the sorted DataFrame to a CSV file (2 different files depending on if this is the entire history or just YoY)
if len(final_df.month.unique()) > 2:
    print("Saving CSV file")
    final_df.to_csv("unique_devices_time_series_analysis.csv", index=False)
else:
    print("Saving CSV file")
    latest_month = final_df['month'].max()
    final_df = final_df[final_df['month'] == latest_month]
    final_df.to_csv("unique_devices_time_series_yoy.csv", index = False)
    
# Show df
final_df

SPARK_HOME: /usr/lib/spark3
Using Hadoop client lib jars at 3.2.0, provided by Spark.
PYSPARK_PYTHON=/opt/conda-analytics/bin/python3


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
23/12/19 20:50:50 WARN SparkConf: Note that spark.local.dir will be overridden by the value set by the cluster manager (via SPARK_LOCAL_DIRS in mesos/standalone/kubernetes and LOCAL_DIRS in YARN).
23/12/19 20:50:51 WARN Utils: Service 'sparkDriver' could not bind on port 12000. Attempting port 12001.
23/12/19 20:50:51 WARN Utils: Service 'sparkDriver' could not bind on port 12001. Attempting port 12002.
23/12/19 20:50:51 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
23/12/19 20:50:51 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
23/12/19 20:50:59 WARN Utils: Service 'org.apache.spark.network.netty.NettyBlockTransferService' could not bind on port 13000. Attempting port 13001.
23/12/19 20:50:59 WARN Utils: Service 'org.apache.spark.network.netty.NettyBlockTransferService' could not bind on 

Saving CSV file


,month,country,wmf_region,current_metric,previous_metric,absolute_change,formula_result_percentage_regional,simple_calculation_regional,rank_simple_calculation,rank_formula_result,simple_calculation_global,formula_result_percentage_global
29,2023-11,Poland,Central & Eastern Europe & Central Asia,23244924.0,24621050.0,1376126.0,16.527084,15.289363,1.0,1.0,0.973112,2.040120
13,2023-11,Bulgaria,Central & Eastern Europe & Central Asia,5233706.0,3960332.0,1273374.0,12.353854,14.147743,2.0,2.0,0.900452,0.826500
39,2023-11,Uzbekistan,Central & Eastern Europe & Central Asia,4899869.0,3675465.0,1224404.0,11.943510,13.603665,3.0,3.0,0.865824,0.800498
21,2023-11,Kazakhstan,Central & Eastern Europe & Central Asia,8714369.0,8080953.0,633416.0,4.615176,7.037529,4.0,7.0,0.447913,0.158872
18,2023-11,Georgia,Central & Eastern Europe & Central Asia,2478170.0,1897210.0,580960.0,5.749386,6.454720,5.0,6.0,0.410819,0.374637
...,...,...,...,...,...,...,...,...,...,...,...,...
153,2023-11,Equatorial Guinea,Sub-Saharan Africa,39357.0,38995.0,362.0,0.012187,0.004763,51.0,48.0,0.000256,0.001321
187,2023-11,São Tomé and Príncipe,Sub-Saharan Africa,7552.0,7278.0,274.0,0.006510,0.003605,52.0,50.0,0.000194,0.000085
157,2023-11,French Southern Territories,Sub-Saharan Africa,2.0,20.0,18.0,0.000367,0.000237,53.0,53.0,0.000013,0.000015
191,2023-11,Western Sahara,Sub-Saharan Africa,154.0,158.0,4.0,0.000063,0.000053,54.0,54.0,0.000003,0.000010


In [2]:
import os
print(os.getcwd())

/srv/home/hghani/hghani
